Anexo A, apartado A.2 · Adaptadores: configuración común de entrenamiento.

Cuaderno con el que se entrenó LoRA-irrelevante en Google Colab y se calculó, en la misma sesión, la
verosimilitud por sesión de Centaur, de LoRA-ruido y de LoRA-irrelevante sobre el corpus de evaluación.


# Stage 7.5.2 — LoRA-irrelevant training + 3-way evaluation (Colab variant)

Train LoRA-irrelevant adapter (mismo config Centaur, corpus académico no-conductual) + 3-way comparison (Centaur vs LoRA-noise vs LoRA-irrelevant) sobre IGT data. Produces H4 FULL verdict.

**Path**: **canonical A100 80GB** (`max_seq_length=32768` from `lora_config.yaml`).
**Prerequisitos**: Stage 7.5.1 completed (lora_noise_adapter present en Drive, mismo canonical path).
**Compute**: ~20h A100 training + ~2h eval = ~22h total. Cabe en 1 session Pro+.
**Cost**: ~$8-10 equivalente.

**Reduced A100 40GB path** (opt-in, NO default):
- En Cell 2: `check_colab_env.py --stage 7.5.2 --reduced-seq-plan`.
- En Cell 4: `train_lora_irrelevant.py ... --max-seq-length 16384`.
- CRÍTICO: Stage 7.5.1 lora_noise_adapter DEBE haberse entrenado con el mismo `max_seq_length` (isolation content vs noise vs irrelevant requires identical architecture+seq).


## Cell 1: Colab env

In [ ]:
# Clone repo + run colab_setup.py
# Si el repo es privado, requiere GH_TOKEN en Colab Secrets (🔑 sidebar):
#   Name: GH_TOKEN, Value: PAT con `Contents: Read-only` para este repo, Notebook access ON.
# Si es public, GH_TOKEN se ignora (clone anónimo).
from google.colab import userdata
import os
gh_token = None
try:
    gh_token = userdata.get('GH_TOKEN')
except Exception:
    pass

if not os.path.exists('/content/ai-system-lab'):
    if gh_token:
        !git clone <REPO> /content/ai-system-lab
    else:
        !git clone <REPO> /content/ai-system-lab
else:
    !cd /content/ai-system-lab && git pull

assert os.path.exists('/content/ai-system-lab/tesis'), 'Clone failed — revisa GH_TOKEN o repo visibility'
!python /content/ai-system-lab/tesis/data_analyses/llm_evaluation/paper_01_igt/launchers/colab/colab_setup.py


## Cell 2: Verify Stage 7.5.1 completed

In [ ]:
# [revisión interna] polish post-launch: load HF_TOKEN directly into notebook kernel
import os
if not os.environ.get('HF_TOKEN'):
    try:
        from google.colab import userdata
        tok = userdata.get('HF_TOKEN')
        if tok:
            os.environ['HF_TOKEN'] = tok
    except Exception as e:
        print(f'WARN: could not read Colab Secret: {e}')
assert os.environ.get('HF_TOKEN'), 'Set HF_TOKEN in Colab Secrets (🔑 sidebar) + toggle Notebook access ON'

# Sanity: lora_noise_adapter from Stage 7.5.1 must exist en Drive
noise_path = f'/content/drive/MyDrive/paper_01_igt_results/stage_7_5_1/lora_noise_adapter'
assert os.path.exists(noise_path), f'Stage 7.5.1 NOT completed: {noise_path} not found'
print(f'✓ Stage 7.5.1 adapter found at {noise_path}')

%cd /content/ai-system-lab
!python tesis/data_analyses/llm_evaluation/paper_01_igt/launchers/colab/check_colab_env.py --stage 7.5.2


## Cell 3: Generate irrelevant corpus (CPU, ~5 min)

In [ ]:
import os
os.makedirs('/content/drive/MyDrive/paper_01_igt_results/stage_7_5_2', exist_ok=True)

%cd tesis/data_analyses/llm_evaluation/paper_01_igt/stage_7_5_2_lora_irrelevant_and_3way_eval/

# Codex consultation R1 (sintético vs real), R2 (HF source + framing),
# R3 (P0 marker-free JSONL bug fix). Spec:
#   - Source = canonical HF Wikipedia snapshot (committed in corpora/, SHA256 archived)
#   - --from-file mode emits marker-wrapped contiguous chunks (no random resampling)
#   - Preflight: assert >=1 <<X>> span per record + collator yields nonzero labels
#   - Framing: coherent non-behavioral content, NOT pretraining-mirror.
DRIVE_OUT = '/content/drive/MyDrive/paper_01_igt_results/stage_7_5_2'

!python generate_irrelevant_corpus.py \
    --from-file corpora/irrelevant_wikipedia_20231101_en_100k_2026-05-19.txt \
    --output {DRIVE_OUT}/corpus_irrelevant.jsonl \
    --output-manifest {DRIVE_OUT}/corpus_irrelevant_manifest.json \
    --tokens 100000 \
    --seed 20260506

# Cell 3b: Codex R3 P0 preflight — DO NOT launch Cell 4 if this fails.
# Cheap substring check + collator semantic check (loads Llama tokenizer).
!python preflight_marker_check.py \
    --jsonl {DRIVE_OUT}/corpus_irrelevant.jsonl \
    --collator-check \
    --base-model meta-llama/Llama-3.1-70B \
    --n-sample 5


## Cell 4: Train LoRA-irrelevant (~20h A100)

In [ ]:
# Reusa el mismo lora_config.yaml de Stage 7.5.1 (isolación content vs noise)
!python train_lora_irrelevant.py \
    --corpus /content/drive/MyDrive/paper_01_igt_results/stage_7_5_2/corpus_irrelevant.jsonl \
    --output_adapter /content/drive/MyDrive/paper_01_igt_results/stage_7_5_2/lora_irrelevant_adapter \
    --config ../stage_7_5_1_lora_noise/lora_config.yaml

## Cell 5: 3-way eval (~2h A100)

In [ ]:
# Bonferroni-2 (α=0.025) sobre 2 paired comparisons:
#   - Centaur vs LoRA-noise (d_vs_noise)
#   - Centaur vs LoRA-irrelevant (d_vs_irrelevant)
!python eval_3way.py \
    --centaur-adapter marcelbinz/Llama-3.1-Centaur-70B-adapter \
    --noise-adapter /content/drive/MyDrive/paper_01_igt_results/stage_7_5_1/lora_noise_adapter \
    --irrelevant-adapter /content/drive/MyDrive/paper_01_igt_results/stage_7_5_2/lora_irrelevant_adapter \
    --igt-data /content/ai-system-lab/tesis/data_analyses/llm_evaluation/paper_01_igt/manifests/igt_paper1_eval_data.json \
    --output /content/drive/MyDrive/paper_01_igt_results/stage_7_5_2/

## Cell 6: H4 verdict interpretation (CPU, ~5s)

In [ ]:
# Maps stats to one of 4 H4 verdicts canónicos:
#   - strong_training / partial_training / confound_lora_overhead / inconclusive
!python interpret_h4_verdict.py \
    --results /content/drive/MyDrive/paper_01_igt_results/stage_7_5_2/eval_3way.json \
    --output /content/drive/MyDrive/paper_01_igt_results/stage_7_5_2/h4_verdict.md

# Display verdict
!cat /content/drive/MyDrive/paper_01_igt_results/stage_7_5_2/h4_verdict.md

## Done

Outputs en `/content/drive/MyDrive/paper_01_igt_results/stage_7_5_2/`:
- `lora_irrelevant_adapter/` — PEFT adapter
- `nll_centaur.json`, `nll_lora_noise.json`, `nll_lora_irrelevant.json` — per-prompt NLL
- `eval_3way.json` — stats Bonferroni-2 + d_vs_noise + d_vs_irrelevant
- `h4_verdict.md` — drop directamente en [revisión interna] [revisión interna]-[revisión interna] prose

Entrenamiento y evaluación del adaptador de control con prosa irrelevante completados.
